#  RT-DETR Pothole Detection
**Real-Time DEtection TRansformer** fine-tuned for pothole detection



In [1]:
# Installs ultralytics, kagglehub, seaborn and tqdm.
import subprocess, sys


def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])


pip_install("ultralytics", "kagglehub", "seaborn", "tqdm")
print("Dependencies installed")

Dependencies installed



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "rtdetr_results")
DATA_DIR = os.path.join(ROOT, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"ROOT       : {ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"SAVE_DIR   : {SAVE_DIR}")

Running on LOCAL JUPYTER
ROOT       : .
OUTPUT_DIR : ./rtdetr_results
SAVE_DIR   : ./saved_models


In [3]:
# Loads all shared imports and confirms GPU availability.
import torch, numpy as np, pandas as pd, cv2
import time, json, glob, warnings, shutil, yaml
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {DEVICE}")
print(f"Torch       : {torch.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"OpenCV      : {cv2.__version__}")

Device      : cuda
Torch       : 2.10.0+cu128
NumPy       : 2.2.6
OpenCV      : 4.13.0


In [4]:
# Defines the fixed evaluation config, CLASS_NAMES, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the SKIP_TRAINING switch.
CLASS_NAMES = ["pothole"]
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = None
SKIP_TRAINING = True  # True = load saved weights, False = train

PALETTE = {
    "YOLOv8m": "#00d4ff",
    "YOLOv10m": "#3b82f6",
    "YOLOv11m": "#6366f1",
    "Faster R-CNN": "#f97316",
    "SSD-VGG16": "#ec4899",
    "YOLO+FRCNN Ensemble": "#a855f7",
    "YOLOv8m+CBAM": "#22c55e",
    "YOLOv8m+CoordAttn": "#eab308",
    "RT-DETR": "#10b981",
}

print("Config loaded")
print(f"   CONF_THRESH  = {CONF_THRESH}")
print(f"   IOU_THRESH   = {IOU_THRESH}")
print(f"   IMG_SIZE     = {IMG_SIZE}")
print(f"   MAX_IMAGES   = {MAX_IMAGES}")
print(f"   SKIP_TRAINING= {SKIP_TRAINING}")

Config loaded
   CONF_THRESH  = 0.25
   IOU_THRESH   = 0.45
   IMG_SIZE     = 640
   MAX_IMAGES   = None
   SKIP_TRAINING= True
